# GPT-SoVITS Voice Training — T4x2 GPU修复了 GPU 未用上的问题。

In [ ]:
# @title Step 1/7: 安装 PyTorch CUDA 版# ⚠️ 运行完后必须重启内核（Runtime → Restart Session）import sys, subprocess, osprint("="*60)print("安装 PyTorch cu124 (Kaggle T4 推荐)")print("="*60)# 卸载可能已存在的 CPU 版subprocess.run(    [sys.executable, "-m", "pip", "uninstall", "-y",     "torch", "torchaudio", "torchvision"],    capture_output=True)# 安装 CUDA 12.4 版本cmd = [    sys.executable, "-m", "pip", "install", "--no-input",    "torch==2.4.0+cu124",    "torchaudio==2.4.0+cu124",    "--index-url", "https://download.pytorch.org/whl/cu124"]subprocess.run(cmd, capture_output=False)print("" + "="*60)print("⚠️  ⚠️  ⚠️  重要：安装完成后，请点击 Kaggle 上方菜单")print("    Runtime → Restart Session（重启内核）")print("    然后再运行 Step 2！")print("="*60)

In [ ]:
# @title Step 2/7: 验证 GPU 可用（重启内核后运行）import torch, osprint("="*60)print("验证 PyTorch 检测到 GPU")print("="*60)print(f"PyTorch 版本: {torch.__version__}")print(f"CUDA 可用: {torch.cuda.is_available()}")if not torch.cuda.is_available():    print("❌ GPU 未检测到！请检查：")    print("  1. Step 1 安装时有没有报错？")    print("  2. 有没有重启内核？")    raise SystemExit("停止：PyTorch 未检测到 GPU")print(f"✅ 检测到 {torch.cuda.device_count()} 个 GPU：")for i in range(torch.cuda.device_count()):    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")# 设置可见 GPUn_gpus = torch.cuda.device_count()os.environ['CUDA_VISIBLE_DEVICES'] = ','.join(str(i) for i in range(n_gpus))print(f"✅ CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")# 放一个张量到 GPU 测试x = torch.randn(5, 5).cuda()print(f"✅ 测试张量设备: {x.device}")

In [ ]:
# @title Step 3/7: 克隆 GPT-SoVITS 并安装依赖import subprocess, os, sysprint("="*60)print("克隆 GPT-SoVITS")print("="*60)os.chdir("/kaggle/working")if not os.path.exists("GPT-SoVITS"):    subprocess.run([        "git", "clone", "--depth=1",        "https://github.com/RVC-Boss/GPT-SoVITS.git"    ], check=True)    print("✅ 克隆完成")else:    print("✅ 已存在，跳过克隆")os.chdir("/kaggle/working/GPT-SoVITS")print("" + "="*60)print("安装依赖...")print("="*60)# 安装 requirements（不要用 capture_output，看有没有报错）subprocess.run(    [sys.executable, "-m", "pip", "install", "--no-input", "-r", "requirements.txt"],    capture_output=False)print("✅ 依赖安装完成")

In [ ]:
# @title Step 4/7: 下载预训练模型import osfrom huggingface_hub import hf_hub_downloadprint("="*60)print("下载预训练模型")print("="*60)os.chdir("/kaggle/working/GPT-SoVITS")gpt_path = hf_hub_download(    repo_id="RVC-Boss/GPT-SoVITS",    filename="pretrained_models/gpt.ckpt",    local_dir="/kaggle/working/GPT-SoVITS")print(f"✅ GPT 模型: {gpt_path}")sovits_path = hf_hub_download(    repo_id="RVC-Boss/GPT-SoVITS",    filename="pretrained_models/sovits.pth",    local_dir="/kaggle/working/GPT-SoVITS")print(f"✅ SoVITS 模型: {sovits_path}")

In [ ]:
# @title Step 5/7: 准备数据集import os, shutil, json, zipfileprint("="*60)print("准备数据集")print("="*60)# ⚠️ 修改这里：你的 Kaggle Dataset 路径INPUT_DIR = "/kaggle/input/your-dataset-name"  # ← 改成你的路径DATASET_DIR = "/kaggle/working/datasets/4-cjw"os.makedirs(DATASET_DIR, exist_ok=True)# 如果是 zip，先解压for f in os.listdir(INPUT_DIR):    if f.endswith(".zip"):        with zipfile.ZipFile(os.path.join(INPUT_DIR, f)) as z:            z.extractall(INPUT_DIR)        print(f"✅ 解压: {f}")# 收集 wav 文件wav_files = []for root, _, files in os.walk(INPUT_DIR):    for f in files:        if f.endswith(".wav"):            wav_files.append(os.path.join(root, f))print(f"✅ 找到 {len(wav_files)} 个 WAV 文件")# 复制到数据集目录（用软链接省空间）for i, wav_path in enumerate(wav_files):    dst = os.path.join(DATASET_DIR, f"{i:05d}.wav")    if not os.path.exists(dst):        os.symlink(wav_path, dst)print(f"✅ 数据集准备完成: {DATASET_DIR}")# 如果有标注文件ann_file = os.path.join(INPUT_DIR, "annotations.json")if os.path.exists(ann_file):    with open(ann_file) as f:        annotations = json.load(f)    for i, ann in enumerate(annotations):        txt_path = os.path.join(DATASET_DIR, f"{i:05d}.txt")        with open(txt_path, "w", encoding="utf-8") as f:            f.write(ann.get("text", ""))    print(f"✅ 标注文件创建完成: {len(annotations)} 条")

In [ ]:
# @title Step 6/7: 训练 SoVITS（GPU）import os, sys, subprocess, torchprint("="*60)print("训练 SoVITS")print("="*60)os.chdir("/kaggle/working/GPT-SoVITS")OUTPUT_DIR = "/kaggle/working/output"os.makedirs(OUTPUT_DIR, exist_ok=True)# 再次确认 GPU 可用print(f"CUDA available: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")# 关键：在 env 里传 CUDA_VISIBLE_DEVICESenv = os.environ.copy()env["CUDA_VISIBLE_DEVICES"] = "0,1" if torch.cuda.device_count() >= 2 else "0"env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"print(f"✅ 使用 GPU: {env['CUDA_VISIBLE_DEVICES']}")# 运行训练（不用 capture_output，看实时进度）cmd = [    sys.executable, "train.py",    "-s", "/kaggle/working/datasets/4-cjw",    "-m", OUTPUT_DIR,    "-n", "cjw",    "--train_type", "sovits",    "--pretrained_sovits", "/kaggle/working/GPT-SoVITS/pretrained_models/sovits.pth",    "--batch_size", "8",    "--epochs", "8",    "--lr", "4e-4",    "--save_every_epoch", "4",]print(f"命令: {' '.join(cmd)}")print("="*60)proc = subprocess.run(cmd, env=env)print(f"训练结束，返回码: {proc.returncode}")

In [ ]:
# @title Step 7/7: 训练 GPT（GPU）import os, sys, subprocess, torchprint("="*60)print("训练 GPT")print("="*60)os.chdir("/kaggle/working/GPT-SoVITS")OUTPUT_DIR = "/kaggle/working/output"env = os.environ.copy()env["CUDA_VISIBLE_DEVICES"] = "0,1" if torch.cuda.device_count() >= 2 else "0"cmd = [    sys.executable, "train.py",    "-s", "/kaggle/working/datasets/4-cjw",    "-m", OUTPUT_DIR,    "-n", "cjw",    "--train_type", "gpt",    "--pretrained_gpt", "/kaggle/working/GPT-SoVITS/pretrained_models/gpt.ckpt",    "--batch_size", "8",    "--epochs", "15",    "--lr", "1e-4",    "--save_every_epoch", "5",]print(f"命令: {' '.join(cmd)}")print("="*60)proc = subprocess.run(cmd, env=env)print(f"✅ 全部训练完成！输出在: {OUTPUT_DIR}")# 列出输出文件print("输出文件：")for f in sorted(os.listdir(OUTPUT_DIR)):    fpath = os.path.join(OUTPUT_DIR, f)    if os.path.isfile(fpath):        size_mb = os.path.getsize(fpath) / 1024 / 1024        print(f"  {f} ({size_mb:.1f} MB)")